# XGB Orchestration

This notebook keeps the forecasting features built from raw 2023 daily usage data.
Cluster labels are only used to route households into cluster-specific XGB models.

In [1]:
from pathlib import Path
import os
import sys
import subprocess

REPO_NAME = "Clustering-And-Forecasting-TimeSeries-PlayingGround"
REPO_URL = "https://github.com/QuirkyCroissant/Clustering-And-Forecasting-TimeSeries-PlayingGround.git"
BRANCH = os.environ.get("NOTEBOOK_GIT_BRANCH", "main")

kaggle_root = Path("/kaggle/working")
cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME and (cwd / "src" / "forecasting").exists():
    REPO_ROOT = cwd
elif kaggle_root.exists():
    repo_dir = kaggle_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    REPO_ROOT = repo_dir.resolve()
else:
    raise RuntimeError(
        "Could not determine REPO_ROOT automatically. Open the notebook from the repo root or run it on Kaggle."
    )

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH)

Cloning into '/kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround'...


REPO_ROOT: /kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround
SRC_PATH: /kaggle/working/Clustering-And-Forecasting-TimeSeries-PlayingGround/src


Updating files: 100% (83/83), done.


In [2]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

from forecasting.data import (
    discover_cluster_cases,
    ensure_experiment_dirs,
    ensure_output_dirs,
    load_wide_csv,
)

OUT = ensure_output_dirs(REPO_ROOT)

TRAIN_23_PATH = REPO_ROOT / "data" / "raw" / "sample_23.csv"
TEST_24_PATH = REPO_ROOT / "data" / "raw" / "sample_24.csv"
CLUSTER_CASE_DIR = REPO_ROOT / "notebooks" / "outputs" / "feature"
SHAPELET_STATIC_PATH = REPO_ROOT / "notebooks" / "outputs" / "shapelet_experiments" / "shapelet_features.csv"

In [3]:
DEBUG = False
DEBUG_FRAC = 0.2

MODEL_NAME = "xgb"
GPU_ENABLED = True
MIN_CLUSTER_ROWS = 500
MIN_CLUSTER_HOUSEHOLDS = 30

SELECTED_CASES = ["case2", "case5"]
INCLUDE_BASE_VARIANT = True
INCLUDE_SHAPELET_STATIC_VARIANT = False

XGB_PROFILE = "regularized"
XGB_PROFILES = ["regularized", "shallow", "deeper"]
XGB_PARAMS = None
RECURSIVE_VALIDATION_ENABLED = True
RECURSIVE_VALIDATION_DAYS = 60
CLUSTER_GATING_ENABLED = True
CLUSTER_GATE_MARGIN = 0.0
INCLUDE_PROFILE_FEATURES = True
INCLUDE_SEASONAL_PRIORS = True

PLOT_SAMPLE_PER_GROUP = 2
PLOT_MAX_GROUPS = None
RANDOM_STATE = 42

In [4]:
def build_experiment_configs(case_paths: dict[str, Path]) -> list[dict]:
    case_names = SELECTED_CASES or list(case_paths)
    missing = sorted(set(case_names) - set(case_paths))
    if missing:
        raise ValueError(f"Unknown case names requested: {missing}")

    variant_defs = []
    if INCLUDE_BASE_VARIANT:
        variant_defs.append(("base", None))
    if INCLUDE_SHAPELET_STATIC_VARIANT:
        if not SHAPELET_STATIC_PATH.exists():
            raise FileNotFoundError(f"Missing shapelet static feature file: {SHAPELET_STATIC_PATH}")
        variant_defs.append(("shapelet_static", SHAPELET_STATIC_PATH))

    configs = []
    for case_name in case_names:
        for variant_name, static_path in variant_defs:
            configs.append(
                {
                    "case_name": case_name,
                    "variant_name": variant_name,
                    "experiment_name": f"{case_name}_{variant_name}",
                    "cluster_path": str(case_paths[case_name]),
                    "static_features_path": str(static_path) if static_path else None,
                }
            )
    return configs


def build_run_settings() -> dict:
    return {
        "debug": DEBUG,
        "debug_frac": DEBUG_FRAC,
        "model_name": MODEL_NAME,
        "gpu_enabled": GPU_ENABLED,
        "min_cluster_rows": MIN_CLUSTER_ROWS,
        "min_cluster_households": MIN_CLUSTER_HOUSEHOLDS,
        "xgb_profile": XGB_PROFILE,
        "xgb_profiles": XGB_PROFILES,
        "xgb_params": XGB_PARAMS,
        "recursive_validation_enabled": RECURSIVE_VALIDATION_ENABLED,
        "recursive_validation_days": RECURSIVE_VALIDATION_DAYS,
        "cluster_gating_enabled": CLUSTER_GATING_ENABLED,
        "cluster_gate_margin": CLUSTER_GATE_MARGIN,
        "include_profile_features": INCLUDE_PROFILE_FEATURES,
        "include_seasonal_priors": INCLUDE_SEASONAL_PRIORS,
        "plot_sample_per_group": PLOT_SAMPLE_PER_GROUP,
        "plot_max_groups": PLOT_MAX_GROUPS,
        "random_state": RANDOM_STATE,
    }


def run_experiment_subprocess(exp_config: dict, run_settings: dict):
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_PATH) if not existing_pythonpath else str(SRC_PATH) + os.pathsep + existing_pythonpath

    cmd = [
        sys.executable,
        "-m",
        "forecasting.experiment",
        "--repo-root",
        str(REPO_ROOT),
        "--config-json",
        json.dumps(exp_config),
        "--settings-json",
        json.dumps(run_settings),
    ]

    try:
        subprocess.run(cmd, check=True, cwd=REPO_ROOT, env=env)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Experiment failed: {exp_config['experiment_name']}") from exc


def load_experiment_metric(exp_name: str, table_name: str) -> pd.DataFrame:
    metric_path = ensure_experiment_dirs(REPO_ROOT, exp_name)["metrics"] / f"{table_name}.csv"
    if not metric_path.exists():
        raise FileNotFoundError(f"Missing metric output for {exp_name}: {metric_path}")
    return pd.read_csv(metric_path)

In [5]:
cluster_case_paths = discover_cluster_cases(CLUSTER_CASE_DIR)
experiment_configs = build_experiment_configs(cluster_case_paths)
run_settings = build_run_settings()

experiment_plan = pd.DataFrame(experiment_configs)
display(experiment_plan)
print(f"Planned experiments: {len(experiment_configs)}")

,case_name,variant_name,experiment_name,cluster_path,static_features_path
0,case2,base,case2_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None
1,case5,base,case5_base,/kaggle/working/Clustering-And-Forecasting-Tim...,None


Planned experiments: 2


In [6]:
completed_experiments = []

for exp_config in experiment_configs:
    print(f"\n=== Running {exp_config['experiment_name']} ===")
    run_experiment_subprocess(exp_config, run_settings)
    completed_experiments.append(exp_config["experiment_name"])


=== Running case2_base ===


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:42:05] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Building training rows: 100%|██████████| 17547/17547 [00:41<00:00, 418.02it/s]


[0]	validation_0-mae:7.55061
[50]	validation_0-mae:3.83816
[100]	validation_0-mae:2.78396
[150]	validation_0-mae:2.57765
[200]	validation_0-mae:2.53543
[250]	validation_0-mae:2.51833
[300]	validation_0-mae:2.50383
[350]	validation_0-mae:2.49329
[400]	validation_0-mae:2.48140
[450]	validation_0-mae:2.47217
[500]	validation_0-mae:2.46216
[550]	validation_0-mae:2.45340
[600]	validation_0-mae:2.44741
[650]	validation_0-mae:2.44142
[700]	validation_0-mae:2.43455
[750]	validation_0-mae:2.42977
[800]	validation_0-mae:2.42381
[850]	validation_0-mae:2.42010
[899]	validation_0-mae:2.41715


Training xgb cluster models:   0%|          | 0/10 [00:00<?, ?it/s]

[0]	validation_0-mae:18.68745
[50]	validation_0-mae:8.41464
[100]	validation_0-mae:6.12453
[150]	validation_0-mae:5.68127
[200]	validation_0-mae:5.59270
[250]	validation_0-mae:5.53699
[300]	validation_0-mae:5.52389
[350]	validation_0-mae:5.51343
[400]	validation_0-mae:5.52134
[450]	validation_0-mae:5.54054
[500]	validation_0-mae:5.53648
[550]	validation_0-mae:5.53968
[600]	validation_0-mae:5.54052
[650]	validation_0-mae:5.54398
[700]	validation_0-mae:5.53664
[750]	validation_0-mae:5.53015
[800]	validation_0-mae:5.51915
[850]	validation_0-mae:5.51694
[899]	validation_0-mae:5.51404


...

Training xgb cluster models:  73%|███████▎  | 8/11 [01:24<00:19,  6.36s/it]

[0]	validation_0-mae:2.05011
[50]	validation_0-mae:1.17437
[100]	validation_0-mae:0.97504
[150]	validation_0-mae:0.92679
[200]	validation_0-mae:0.91196
[250]	validation_0-mae:0.90438
[300]	validation_0-mae:0.90078
[350]	validation_0-mae:0.89626
[400]	validation_0-mae:0.89153
[450]	validation_0-mae:0.88881
[500]	validation_0-mae:0.88573
[550]	validation_0-mae:0.88437
[600]	validation_0-mae:0.88180
[650]	validation_0-mae:0.87921
[700]	validation_0-mae:0.87705
[750]	validation_0-mae:0.87460
[800]	validation_0-mae:0.87336
[850]	validation_0-mae:0.87263
[899]	validation_0-mae:0.87206
[0]	validation_0-mae:13.94068


Training xgb cluster models:  82%|████████▏ | 9/11 [01:27<00:10,  5.48s/it]

[50]	validation_0-mae:5.81671
[100]	validation_0-mae:3.92702
[150]	validation_0-mae:3.67242
[200]	validation_0-mae:3.62640
[250]	validation_0-mae:3.60145
[300]	validation_0-mae:3.57829
[350]	validation_0-mae:3.56194
[400]	validation_0-mae:3.57578
[450]	validation_0-mae:3.58629
[500]	validation_0-mae:3.58081
[550]	validation_0-mae:3.58116
[600]	validation_0-mae:3.59321
[650]	validation_0-mae:3.59781
[700]	validation_0-mae:3.60144
[750]	validation_0-mae:3.60475
[800]	validation_0-mae:3.61090
[850]	validation_0-mae:3.61683
[899]	validation_0-mae:3.62050


Forecasting by cluster: 100%|██████████| 366/366 [04:23<00:00,  1.39it/s]


In [7]:
all_overall_summary = pd.concat(
    [load_experiment_metric(exp_name, "overall_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_recursive_validation_summary = pd.concat(
    [load_experiment_metric(exp_name, "recursive_validation_summary") for exp_name in completed_experiments],
    ignore_index=True,
)

model_comparison = (
    all_overall_summary
    .pivot(index="experiment_name", columns="model", values="mean_mae")
    .reset_index()
)
model_comparison["delta_cluster_minus_global"] = (
    model_comparison[f"cluster_{MODEL_NAME}"] - model_comparison[f"global_{MODEL_NAME}"]
)
model_comparison = model_comparison.sort_values("delta_cluster_minus_global").reset_index(drop=True)

all_overall_summary.to_csv(OUT["metrics"] / "all_experiments_overall_summary.csv", index=False)
model_comparison.to_csv(OUT["metrics"] / "all_experiments_model_comparison.csv", index=False)
all_recursive_validation_summary.to_csv(OUT["metrics"] / "all_experiments_recursive_validation_summary.csv", index=False)

display(all_overall_summary.sort_values(["experiment_name", "model"]).reset_index(drop=True))
display(model_comparison)
display(all_recursive_validation_summary.sort_values("selected_score").reset_index(drop=True))


,n_households,mean_mae,median_mae,std_mae,mean_mape,median_mape,std_mape,model,experiment_name
0,17547,3.379981,2.085116,4.364546,329.335986,37.474228,6237.280214,cluster_xgb,case2_base
1,17547,3.398794,2.103461,4.338784,320.146661,38.176924,5044.374260,global_xgb,case2_base
2,17547,3.368432,2.071450,4.330635,308.308952,37.516991,5062.335865,cluster_xgb,case5_base
3,17547,3.413474,2.099406,4.405548,315.009034,38.179399,4997.504990,global_xgb,case5_base


model,experiment_name,cluster_xgb,global_xgb,delta_cluster_minus_global
0,case5_base,3.368432,3.413474,-0.045041
1,case2_base,3.379981,3.398794,-0.018814


,xgb_profile,validation_days,validation_start,validation_end,global_mae,cluster_mae,gated_cluster_mae,selected_score,n_features,n_trained_cluster_models,n_allowed_cluster_models,experiment_name
0,shallow,60,2023-11-02,2023-12-31,4.414310,4.474053,4.358093,4.358093,80,9,4,case5_base
1,shallow,60,2023-11-02,2023-12-31,4.390325,4.507289,4.387534,4.387534,80,5,3,case2_base
2,regularized,60,2023-11-02,2023-12-31,4.480757,4.552854,4.463631,4.463631,80,9,4,case5_base
3,deeper,60,2023-11-02,2023-12-31,4.492372,4.528207,4.466793,4.466793,80,9,3,case5_base
4,regularized,60,2023-11-02,2023-12-31,4.485794,4.556055,4.485794,4.485794,80,5,0,case2_base
5,deeper,60,2023-11-02,2023-12-31,4.533285,4.532119,4.501036,4.501036,80,5,2,case2_base
